## 这一章准备在初始 LangSmith 的基础上，加上 RAG 相关的内容，用来学习如何进行观测 RAG

### 1.加载环境变量

In [7]:
# 第一步，需要安装 langchain-text-splitters 这个依赖，这个是 langchain 官方内部写的文件分片依赖
from dotenv import find_dotenv, load_dotenv

# usecwd：控制搜索 `.env` 的起始目录
# `usecwd=False`（默认）：从调用 `find_dotenv()` 的那个 py 文件所在目录开始，逐级向上找 `.env`。
# `usecwd=True`：从程序运行时的当前工作目录 `os.getcwd()` 开始向上找 `.env`。
env_path = find_dotenv('.env', usecwd=True)

if not env_path:
    # raise：抛出异常，不是打印日志；不捕获的话程序直接退出，类似 java 中的 throw
    raise FileNotFoundError("从当前目录向上没有找到 .env")

load_dotenv(env_path, override=True)

import os

required_env = [
    "LLM_MODEL",
    "LLM_BASE_URL",
    "LLM_API_KEY",
    "EMBEDDING_MODEL",
    "LANGSMITH_API_KEY"
]

missing = [name for name in required_env if not os.getenv(name)]
if missing:
    raise RuntimeError(f"缺少环境变量：{missing}")

print("已加载：", env_path)
print("聊天模型：", os.getenv("LLM_MODEL"))
print("向量模型：", os.getenv("EMBEDDING_MODEL"))
print("LangSmith 项目：", os.getenv("LANGSMITH_PROJECT"))

已加载： E:\AgentProject\studyAgent\.env
聊天模型： qwen3.7-flash-2026-07-15
向量模型： qwen3.7-text-embedding-flash
LangSmith 项目： study


### 2.加载 Markdown 文档

In [8]:
from pathlib import Path
from langchain_core.documents import Document

knowledge_dir = Path.cwd() / "knowledge"

if not knowledge_dir.exists():
    # 当 Notebook 内核工作目录是项目根目录时使用这个路径
    knowledge_dir = Path.cwd() / "02-rag" / "knowledge"

# knowledge_dir.glob("*.md")：Pathlib 的 glob，返回生成器，遍历目录下所有 .md 文件对象；
markdown_files = sorted(knowledge_dir.glob("*.md"))

if not markdown_files:
    raise FileNotFoundError(f"知识库目录中没有 Markdown 文件：{knowledge_dir}")

documents = []

for file_path in markdown_files:
    content = file_path.read_text(encoding="utf-8")
    documents.append(
        Document(
            page_content=content,
            metadata={"source": file_path.name},
        )
    )

print(f"加载了 {len(documents)} 份文档")
for document in documents:
    print(document.metadata, len(document.page_content), "字符")

加载了 1 份文档
{'source': 'company-handbook.md'} 232 字符


### 3.文本切块

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=50,
    length_function=len,
    separators=["\n## ", "\n### ", "\n\n", "\n", "。", "！", "？", " ", ""],
)

chunks = text_splitter.split_documents(documents)

# enumerate：遍历 chunks，同时拿到下标 + 元素本身
# enumerate(可迭代对象, start=0)
# start 可以指定起始编号，比如 enumerate(chunks, start=1) 序号从 1 开始。
for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = index
    print(f"\n--- chunk {index} / {chunk.metadata['source']} ---")
    print(chunk.page_content)

print(f"\n共生成 {len(chunks)} 个文本块")


--- chunk 0 / company-handbook.md ---
# 星河公司的员工手册

## 年假制度

正式员工每年享有 10 天带薪年假。入职不满一年的员工，年假按照实际在职月份折算。年假申请需要至少提前 3 个工作日提交。

--- chunk 1 / company-handbook.md ---
## 报销制度

单笔金额低于 500 元的普通办公费用由直属主管审批。单笔金额达到或超过 500 元时，还需要财务负责人审批。报销申请必须在费用发生后的 30 天内提交。

--- chunk 2 / company-handbook.md ---
## 远程办公

员工每周最多可以申请 2 天远程办公。远程办公需要提前一天在内部系统提交申请，并获得直属主管批准。

共生成 3 个文本块


### 4.创建 Embedding 模型

In [10]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model=os.getenv("EMBEDDING_MODEL"),
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
    # 百炼兼容接口只接受字符串或字符串列表。
    # 关闭后 LangChain 会发送原始文本，而不是 OpenAI token ID 数组。
    check_embedding_ctx_length=False,
    # 明确要求返回浮点数组，避免第三方兼容接口的 base64 差异。
    encoding_format="float",
)

test_vector = embeddings.embed_query("员工有多少天年假？")

print("向量维度：", len(test_vector))
print("前 5 个数字：", test_vector[:5])

E:\AgentProject\studyAgent\.venv\Lib\site-packages\langchain_openai\embeddings\base.py:359: UserWarning: WARNING! encoding_format is not default parameter.
                    encoding_format was transferred to model_kwargs.
                    Please confirm that encoding_format is what you intended.
  warnings.warn(


向量维度： 1024
前 5 个数字： [-0.00736236572265625, -0.0179901123046875, 0.0251617431640625, 0.04766845703125, -0.05072021484375]


### 5 建立内存向量库

In [11]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
)

print(f"已将 {len(chunks)} 个文本块写入内存向量库")

已将 3 个文本块写入内存向量库


### 6 先单独测试检索

In [12]:
question = "普通办公费用 800 元需要谁审批？"

results = vector_store.similarity_search_with_score(question, k=3)

for rank, (document, score) in enumerate(results, start=1):
    print(f"\n--- 第 {rank} 名，score={score:.4f} ---")
    print("来源：", document.metadata)
    print(document.page_content)


--- 第 1 名，score=0.7097 ---
来源： {'source': 'company-handbook.md', 'chunk_id': 1}
## 报销制度

单笔金额低于 500 元的普通办公费用由直属主管审批。单笔金额达到或超过 500 元时，还需要财务负责人审批。报销申请必须在费用发生后的 30 天内提交。

--- 第 2 名，score=0.5063 ---
来源： {'source': 'company-handbook.md', 'chunk_id': 2}
## 远程办公

员工每周最多可以申请 2 天远程办公。远程办公需要提前一天在内部系统提交申请，并获得直属主管批准。

--- 第 3 名，score=0.3480 ---
来源： {'source': 'company-handbook.md', 'chunk_id': 0}
# 星河公司的员工手册

## 年假制度

正式员工每年享有 10 天带薪年假。入职不满一年的员工，年假按照实际在职月份折算。年假申请需要至少提前 3 个工作日提交。


### 7 将检索封装为可观测函数

In [13]:
from langsmith import traceable


@traceable(run_type="retriever", name="retrieve_company_handbook")
def retrieve(question: str, k: int = 3):
    return vector_store.similarity_search(question, k=k)

### 8 创建聊天模型和 RAG Prompt

In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL"),
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
    temperature=0,
)

rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """你是一个严格依据公司知识库回答问题的助手。

规则：
1. 只能依据下面提供的知识库上下文回答。
2. 如果上下文不足以回答，明确说“根据当前知识库无法确定”。
3. 不要用自己的常识补充公司制度。
4. 回答后列出使用到的来源文件。

知识库上下文：
{context}""",
    ),
    ("human", "{question}"),
])

### 9 完成 RAG 问答函数

In [16]:
from langchain_core.tracers.langchain import wait_for_all_tracers


def format_context(retrieved_documents) -> str:
    sections = []

    for index, document in enumerate(retrieved_documents, start=1):
        source = document.metadata.get("source", "unknown")
        chunk_id = document.metadata.get("chunk_id", "unknown")
        sections.append(
            f"[资料 {index} | source={source} | chunk={chunk_id}]\n"
            f"{document.page_content}"
        )

    return "\n\n".join(sections)


@traceable(name="two_step_rag")
def ask_rag(question: str) -> dict:
    retrieved_documents = retrieve(question, k=3)
    context = format_context(retrieved_documents)
    messages = rag_prompt.invoke({
        "question": question,
        "context": context,
    })
    response = llm.invoke(messages)

    return {
        "answer": response.content,
        "sources": [document.metadata for document in retrieved_documents],
        "context": context,
    }


result = ask_rag("普通办公费用 800 元需要谁审批？")

print("回答：")
print(result["answer"])
print("\n检索来源：")
for source in result["sources"]:
    print(source)

# Notebook 退出或马上打开 LangSmith 查看时，等待后台 trace 上传完成
wait_for_all_tracers()

回答：
根据知识库中的报销制度规定，单笔金额达到或超过 500 元的普通办公费用，在由直属主管审批的基础上，**还需要财务负责人审批**。因此，800 元的普通办公费用需要**直属主管**和**财务负责人**共同审批。

使用到的来源文件：
- company-handbook.md

检索来源：
{'source': 'company-handbook.md', 'chunk_id': 1}
{'source': 'company-handbook.md', 'chunk_id': 2}
{'source': 'company-handbook.md', 'chunk_id': 0}
